# 06 Herbal screening

Put the whole screen together. Pull the compounds reported for a short list of Indonesian medicinal plants out of COCONUT 2.0, check each one against Model A's applicability domain, and run Model A to get condition 1.

Then pair each herb compound with the drugs from notebook 05 and assign a tier with `combine.py`. High, Medium, or Unknown, with the continuous score used only to order pairs inside a tier. The output is a ranked list of hypotheses to test in the lab, nothing more.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "src")))

import numpy as np
import pandas as pd
import joblib

import features
import applicability
import combine
import coconut
import explain

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")

## COCONUT filteringThe bulk download is a ~240 MB zip holding a ~700 MB CSV, so `src/coconut.py`streams it in chunks rather than loading it whole.Species naming is the real difficulty, not the file size. COCONUT's `organisms`field is free text contributed from many source databases, so one plant appearsunder its accepted name, its synonyms and several spellings. `JAMU_SPECIES`lists the variants for each plant; *Curcuma domestica* and *Curcuma zanthorrhiza*are the ones that would otherwise cost the most rows.

In [ ]:
for canonical, variants in coconut.JAMU_SPECIES.items():
    print(f"{canonical}")
    for v in variants:
        print(f"    matches: {v}")

In [ ]:
# Locate the COCONUT bulk file. Nothing pins a release name: COCONUT reissues
# the file monthly and the date is in the filename, so a hard coded name breaks
# silently the moment a newer release is downloaded. Zip or unzipped both work.
#   Download: https://coconut.naturalproducts.net/download  -> data/raw/
COCONUT_BULK = coconut.find_bulk()
print("using COCONUT release:", os.path.basename(COCONUT_BULK))

HERB_CSV = os.path.join(PROCESSED_DIR, "herb_compounds.csv")

# The scan takes a few minutes, so it is cached. Delete herb_compounds.csv to
# force a re-run after editing JAMU_SPECIES.
if os.path.exists(HERB_CSV):
    herb_df = pd.read_csv(HERB_CSV)
    print(f"loaded cached {HERB_CSV}: {len(herb_df)} compounds")
else:
    herb_df, stats = coconut.filter_bulk(COCONUT_BULK)
    herb_df.to_csv(HERB_CSV, index=False)
    print(f"rows scanned:     {stats['rows_scanned']:,}")
    print(f"unique compounds: {stats['unique_compounds']:,}")

print()
print("compounds per plant (a compound shared by two plants counts under each):")
print(coconut.species_counts(herb_df).to_string())

## Fingerprint the herb compounds

In [ ]:
herb_fps, kept = features.smiles_list_to_matrix(herb_df["canonical_smiles"].tolist())
dropped = len(herb_df) - len(kept)

print("herb fingerprints:", herb_fps.shape)
print(f"unparseable SMILES dropped: {dropped}")
if dropped:
    print("dropped identifiers:",
          herb_df.iloc[[i for i in range(len(herb_df)) if i not in set(kept)]]
          ["identifier"].tolist()[:20])

# Keep herb_df and herb_fps in step. Everything downstream indexes them
# together, so a mismatch here would silently attach the wrong compound name
# to the wrong prediction.
herb_df = herb_df.iloc[kept].reset_index(drop=True)
assert len(herb_df) == herb_fps.shape[0]

## Applicability domainThe domain was fitted on Model A's **training** fingerprints in notebook 02.Expect a fair number of these compounds to fall outside it. Plant naturalproducts are structurally unlike the synthetic compounds in `CYP3A4_Veith`, andreporting that honestly is the point of the Unknown tier. A screen that claimedconfident answers for all of them would be the less honest result.

In [ ]:
domain = applicability.load_domain(os.path.join(PROCESSED_DIR, "domain_a.npz"))
# The domain works on plain 0/1 fingerprints, while herb_fps holds fragment
# COUNTS plus ten descriptors, which is what Model A eats. Passing herb_fps
# straight in would not even line up: the domain was fitted on 2048 bit
# columns and herb_fps has 2058. Beyond the shape, Tanimoto is a set
# measure, so counts and molecular weight have no meaning in it.
herb_bits = features.bits_from_matrix(herb_fps)
verdicts, scores = applicability.check_molecules(herb_bits, domain)

herb_df["domain_status"] = verdicts
herb_df["domain_score"] = scores

n_outside = sum(1 for v in verdicts if v == applicability.OUTSIDE)
print(f"domain floor (from Model A training data): {domain['floor']:.4f}")
print(f"herb compounds inside:  {len(verdicts) - n_outside}")
print(f"herb compounds outside: {n_outside}  ({n_outside / len(verdicts):.1%})")
print()
print("Outside the domain -> Unknown tier. Model A is not run on those at all,")
print("because a number the tier logic then discards tends to get quoted anyway.")

## Condition 1: Model A on the compounds inside the domain

In [ ]:
model_a = joblib.load(os.path.join(PROCESSED_DIR, "model_a.joblib"))
model_b = joblib.load(os.path.join(PROCESSED_DIR, "model_b.joblib"))

inside_mask = np.array([v == applicability.INSIDE for v in verdicts])
if inside_mask.any():
    inside_probs = model_a.predict_proba(herb_fps[inside_mask])[:, 1]
    print(f"Model A run on {inside_mask.sum()} in-domain compounds")
    print(f"predicted inhibitors at p >= {combine.HIGH_CONF}: "
          f"{(inside_probs >= combine.HIGH_CONF).sum()}")
    print(f"borderline ({combine.LOW_CONF} - {combine.HIGH_CONF}): "
          f"{((inside_probs > combine.LOW_CONF) & (inside_probs < combine.HIGH_CONF)).sum()}")
    print(f"predicted non-inhibitors at p <= {combine.LOW_CONF}: "
          f"{(inside_probs <= combine.LOW_CONF).sum()}")
else:
    print("no compounds inside the domain")

## Screen the pairs

In [ ]:
drug_df = pd.read_csv(os.path.join(PROCESSED_DIR, "screen_drugs.csv"))
drug_df = drug_df.rename(columns={"name_fda": "name"})

substrates = pd.read_csv(os.path.join(PROCESSED_DIR, "cyp3a4_substrates.csv"))
qt = pd.read_csv(os.path.join(PROCESSED_DIR, "qt_drugs.csv"))
substrate_lookup = combine.build_substrate_lookup(substrates)
qt_lookup = combine.build_qt_lookup(qt)

# Drug fingerprints, for any substrate not on the QT list.
drug_smiles_path = os.path.join(RAW_DIR, "drug_smiles.csv")
drug_fps = None
if os.path.exists(drug_smiles_path):
    ds = pd.read_csv(drug_smiles_path)
    smiles_by_name = dict(zip(ds["name"].astype(str), ds["smiles"].astype(str)))
    rows = []
    for name in drug_df["name"]:
        smi = smiles_by_name.get(str(name), "")
        fp = features.smiles_to_fp(smi) if smi and smi != "nan" else None
        rows.append(fp if fp is not None else np.zeros(features.FP_SIZE, dtype=np.uint8))
    drug_fps = np.vstack(rows)

results = combine.screen_pairs(
    herb_df, drug_df, model_a, model_b,
    substrate_lookup, qt_lookup, domain,
    herb_fps=herb_fps, drug_fps=drug_fps,
)

print(f"flagged pairs: {len(results)}")
print()
print(results["tier"].value_counts().to_string())

In [ ]:
results.to_csv(os.path.join(PROCESSED_DIR, "screening_results.csv"), index=False)
print("saved screening_results.csv")
print()
print("top 20 pairs (tier first, then rank score inside the tier):")
cols = ["herb_name", "drug", "tier", "cond1_prob", "cond2_strength",
        "cond3_source", "cond3_risk_category", "rank_score"]
print(results[cols].head(20).to_string(index=False))

## External check against HerbCYPSource: [github.com/Gumgo91/HerbCYP](https://github.com/Gumgo91/HerbCYP). Theypublish their CYP3A4 predictions, so comparing on the compounds that appear inboth sets is a free external check.Agreement is weak independent support. **Disagreement is reported as is**, notquietly dropped, because neither set of predictions is ground truth.

In [ ]:
# TODO once their prediction file is downloaded into data/raw/:
#   1. load it, match on InChIKey rather than on name or SMILES string
#      (the same molecule has many valid SMILES, so a string join would
#       under-report agreement)
#   2. report the confusion matrix between our call and theirs
#   3. report the compounds where the two disagree, with both probabilities

## Fragments behind the top pairs

In [ ]:
# TODO for the top few pairs, show the fragments that drove Model A:
#   1. explain.explain_model on a sample of the Model A training data for top bits
#   2. explain.bit_to_atoms on the herb compound's SMILES
#   3. explain.describe_fragment and explain.draw_fragment
# so each flagged pair comes with a chemical reason rather than just a number.

## Reading the output

Every row is a hypothesis for laboratory testing. A High tier row does not say a jamu product is dangerous: jamu is a multi-compound extract, and a compound appearing in a plant says nothing about how much of it is in a product or how much reaches the liver. The limitations section of the README applies to this table in full.